# Week 8 Lab: Evaluating a Classifier

We'll evaluate a logistic-regression model that predicts who survived the Titanic. Accuracy alone hides a lot, so we'll look at *which* mistakes the model makes, how the decision **threshold** trades one kind of mistake for another, and how ROC curves summarize every threshold at once.

**Core**: Exercises 1–3 (about 35 minutes). **Extension**: Exercise 4, if you finish early.

Cross-validation is practiced in this week's weekly assignment, so here we stick to a single train/test split. Run the setup cell first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             precision_recall_curve, roc_curve, roc_auc_score)

# The file is sorted by passenger class, so shuffle the rows once; otherwise any
# cross-validation fold would contain only one class of passenger.
titanic = pd.read_csv('data/titanic.csv').sample(frac=1, random_state=42)
X = titanic[['pclass', 'sex', 'age', 'fare']]
y = titanic['survived']          # 1 = survived (the "positive" class)

preprocessor = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), ['age', 'fare']),
    ('cat', OneHotEncoder(), ['pclass', 'sex']),
])
model = make_pipeline(preprocessor, LogisticRegression())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)               # 0/1 predictions (threshold 0.5)
y_proba = model.predict_proba(X_test)[:, 1]  # predicted probability of survival

print(f"{len(X_train)} training and {len(X_test)} test passengers; {y.mean():.0%} survived overall")

#### **Exercise 1**

**Which mistakes does the model make?**

1. Print the confusion matrix for `y_test` and `y_pred`. scikit-learn lays it out with the truth on the rows and the prediction on the columns:

   |  | predicted 0 | predicted 1 |
   |---|---|---|
   | **actual 0** | TN | FP |
   | **actual 1** | FN | TP |

2. Print precision, recall, and F1 using `precision_score`, `recall_score`, and `f1_score`.
3. Now compute precision and recall **yourself** from the four numbers in the confusion matrix, and check that they match.

In [ ]:
# YOUR CODE HERE

**Questions**

1. In plain words, for *this* problem: what does precision measure, and what does recall measure?
2. Which kind of mistake does the model make more often: false positives or false negatives?

_Answer here_

#### **Exercise 2**

**Moving the threshold.** `model.predict` labels a passenger "survived" when the predicted probability is at least **0.5**. Nothing forces us to use 0.5: we can apply any threshold to `y_proba` ourselves.

1. Complete the loop so that, for each threshold, it makes predictions with `(y_proba >= t).astype(int)` and prints precision and recall.
2. Run the plotting cell below it, which shows precision and recall across *every* threshold.

In [ ]:
for t in [0.3, 0.5, 0.7]:
    # YOUR CODE HERE
    pass

In [ ]:
# GIVEN: precision and recall at every possible threshold (adapted from the lecture)
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

plt.figure(figsize=(8, 4))
plt.plot(thresholds, precisions[:-1], "b--", label="Precision", linewidth=2)
plt.plot(thresholds, recalls[:-1], "g-", label="Recall", linewidth=2)
plt.axvline(0.5, color="k", linestyle=":", label="default threshold (0.5)")
plt.xlabel("Threshold")
plt.legend(loc="center left")
plt.grid(True)
plt.show()

**Questions**

1. As the threshold goes up, what happens to precision and to recall? Explain *why* in terms of how many passengers get labeled "survived."
2. Describe one real application where you would choose a **low** threshold, and one where you would choose a **high** threshold. What mistake are you trying to avoid in each?

_Answer here_

#### **Exercise 3**

**ROC curves and AUC.** A ROC curve plots the true positive rate (recall) against the false positive rate across all thresholds, so it summarizes Exercise 2's whole tradeoff in one picture. The **AUC** (area under the curve) is 1.0 for a perfect ranking and 0.5 for random guessing.

The cell below compares our model with a much weaker one that only knows each passenger's fare. Run it, then answer the questions.

In [ ]:
# GIVEN: a weak model that uses only the fare
weak = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression())
weak.fit(X_train[['fare']], y_train)
weak_proba = weak.predict_proba(X_test[['fare']])[:, 1]

plt.figure(figsize=(6, 6))
for name, proba in [("full model", y_proba), ("fare only", weak_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.2f})")
plt.plot([0, 1], [0, 1], "k:", label="random guessing")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate (recall)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

**Questions**

1. Which model is better, and how can you tell from the *shape* of the curves, not just the AUC numbers?
2. What would it mean for a model's curve to lie right on the dotted diagonal?

_Answer here_

---

### Extension exercise (optional)

#### **Exercise 4** (extension)

**Learning curves.** Does the model need more data? Use `sklearn.model_selection.learning_curve` to train `model` on increasing fractions of the data, and plot the mean training and validation scores against training-set size. Do the two curves converge? What does that suggest about collecting more passengers' data?

In [ ]:
from sklearn.model_selection import learning_curve

# YOUR CODE HERE
# Hint: learning_curve(model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 8)) returns
# (train_sizes, train_scores, val_scores); average the scores over axis=1 before plotting.